# 🎯 ISSUES + LiLT — PEFT/LoRA Fine-tuning (Reduced Complexity)
> **Architecture: CLIP Visual + Phi TextualInversion + LiLT Layout Branch**  
> LiLT replaces LayoutLMv3 — no redundant vision pathway, ~70% fewer layout-branch params  
> RoBERTa text side of LiLT: **frozen** · Layout Transformer: fine-tuned with LoRA (r=4)  
> All projection/combiner layers trained from scratch · Phi weights frozen (used as-is)

### Key changes vs LayoutLMv3 version
| | LayoutLMv3 + LoRA | **LiLT + LoRA** |
|---|---|---|
| Layout encoder params | 125M | ~22M (layout-only stream) |
| LoRA rank | r=16 | **r=4** |
| Takes `pixel_values`? | ✅ Yes (redundant with CLIP) | ❌ No |
| Text encoder frozen? | ❌ Partial | ✅ Fully frozen |
| Collator: layout processor | LayoutLMv3Processor | **RobertaTokenizer + bbox only** |


In [1]:
# import os
# import shutil

# working_dir = "/kaggle/working"

# for item in os.listdir(working_dir):
#     item_path = os.path.join(working_dir, item)
    
#     try:
#         if os.path.isfile(item_path) or os.path.islink(item_path):
#             os.unlink(item_path)  # delete file
#         elif os.path.isdir(item_path):
#             shutil.rmtree(item_path)  # delete folder
#     except Exception as e:
#         print(f"Failed to delete {item_path}: {e}")

# print("✅ /kaggle/working directory cleared.")

In [2]:
# import gc
# import torch

# # Clear PyTorch cache
# torch.cuda.empty_cache()

# # Collect garbage
# gc.collect()

# # Clear IPC collected memory
# torch.cuda.ipc_collect()

# print("✅ GPU memory cleared")

In [3]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "ftfy", "regex", "tqdm",
    "git+https://github.com/openai/CLIP.git",
    "pytorch-lightning==2.2.4",
    "torchmetrics==1.3.2",
    "scikit-learn",
    "transformers>=4.35.0",
    "peft>=0.10.0",
    "pytesseract",
    "Pillow",
    "easyocr",
    "seaborn",
)
print("✅ All packages installed.")


✅ All packages installed.


In [4]:
import os, json, random, warnings, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from pathlib import Path
from tqdm import tqdm

import clip
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from torch.utils.data import Dataset, DataLoader

# LiLT imports (no LayoutLMv3Processor — LiLT uses standard RoBERTa tokenizer)
from transformers import (
    LiltModel,
    RobertaTokenizerFast,
)
from peft import LoraConfig, get_peft_model, TaskType

from torchvision import transforms as T
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, precision_recall_curve,
    average_precision_score, precision_recall_fscore_support,
)

warnings.filterwarnings('ignore')
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Device: {device}")
if device == 'cuda':
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


🖥️  Device: cuda
   GPU : Tesla T4
   VRAM: 15.6 GB


In [5]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    # ── Paths ─────────────────────────────────────────────────────────────
    dataset_root:        str  = '/kaggle/input/datasets/williamberrios/hateful-memes'
    phi_weights_path:    str  = '/kaggle/input/datasets/arunjatothu/phi-imagenet-45-2/phi_imagenet_45.pt'
    proj_weights_path:   str  = '/kaggle/input/datasets/arunjatothu/hmc-1024-projection-embeddings/hmc_1024_projection_embeddings.pt'
    # LiLT model — RoBERTa-base paired layout transformer (English)
    lilt_model:          str  = 'SCUT-DLVCLab/lilt-roberta-en-base'
    output_dir:          str  = '/kaggle/working/outputs'
    checkpoint_dir:      str  = '/kaggle/working/checkpoints'
    experiment_log_dir:  str  = '/kaggle/working/experiments'

    # ── Data ──────────────────────────────────────────────────────────────
    dataset:             str  = 'hmc'
    image_size:          int  = 224
    fast_process:        bool = False

    # ── Model ─────────────────────────────────────────────────────────────
    name:                str  = 'issues-layout'
    num_mapping_layers:  int  = 1
    map_dim:             int  = 1024
    fusion:              str  = 'align'
    num_pre_output_layers: int = 1
    drop_probs:          List[float] = field(default_factory=lambda: [0.3, 0.4, 0.4])
    proj_map:            bool = False
    pretrained_proj_weights: bool = False
    freeze_proj_layers:  bool = False
    comb_proj:           bool = False
    comb_fusion:         str  = 'align'
    convex_tensor:       bool = False
    text_inv_proj:       bool = False
    phi_inv_proj:        bool = False
    post_inv_proj:       bool = True
    enh_text:            bool = True
    phi_freeze:          bool = True

    pos_weight:          float = 1.5

    # ── LiLT-specific LoRA config ──────────────────────────────────────────
    # r=4 is sufficient — LiLT layout Transformer is already compact (~22M params)
    # RoBERTa text side is fully frozen; only layout attention gets LoRA
    lora_r:              int   = 4
    lora_alpha:          int   = 16
    lora_dropout:        float = 0.05
    # LiLT layout attention projections live under 'layout_*' named modules
    lora_target_modules: List[str] = field(
        default_factory=lambda: ["layout_query", "layout_key", "layout_value"])

    layout_dim:          int   = 256
    layout_dropout:      float = 0.3
    use_ocr_cache:       bool  = True
    ocr_cache_dir:       str   = '/kaggle/working/ocr_cache'
    lilt_max_length:     int   = 512

    # ── Training ──────────────────────────────────────────────────────────
    batch_size:          int   = 16
    lr:                  float = 3e-5
    weight_decay:        float = 5e-3
    max_epochs:          int   = 30
    gradient_clip_val:   float = 0.5
    label_smoothing:     float = 0.05
    log_every_n_steps:   int   = 50
    val_check_interval:  float = 1.0
    limit_train_batches: float = 1.0
    limit_val_batches:   float = 1.0
    experiment_name:     str   = 'issues_lilt_lora_v1'

args = Config()

for d in [args.output_dir, args.checkpoint_dir, args.experiment_log_dir, args.ocr_cache_dir]:
    os.makedirs(d, exist_ok=True)

print("✅ Configuration loaded.")
for k, v in vars(args).items():
    print(f"   {k:35s}: {v}")


✅ Configuration loaded.
   dataset_root                       : /kaggle/input/datasets/williamberrios/hateful-memes
   phi_weights_path                   : /kaggle/input/datasets/arunjatothu/phi-imagenet-45-2/phi_imagenet_45.pt
   proj_weights_path                  : /kaggle/input/datasets/arunjatothu/hmc-1024-projection-embeddings/hmc_1024_projection_embeddings.pt
   lilt_model                         : SCUT-DLVCLab/lilt-roberta-en-base
   output_dir                         : /kaggle/working/outputs
   checkpoint_dir                     : /kaggle/working/checkpoints
   experiment_log_dir                 : /kaggle/working/experiments
   dataset                            : hmc
   image_size                         : 224
   fast_process                       : False
   name                               : issues-layout
   num_mapping_layers                 : 1
   map_dim                            : 1024
   fusion                             : align
   num_pre_output_layers             

In [6]:
class FocalBCELoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.smoothing = smoothing

    def forward(self, logits, labels):
        eps    = self.smoothing
        labels = labels * (1 - eps) + 0.5 * eps
        bce    = F.binary_cross_entropy_with_logits(logits, labels, reduction='none')
        prob   = torch.sigmoid(logits)
        pt     = torch.where(labels >= 0.5, prob, 1 - prob)
        focal_weight = (1 - pt) ** self.gamma
        alpha_t = torch.where(labels >= 0.5,
                              torch.full_like(labels, self.alpha),
                              torch.full_like(labels, 1 - self.alpha))
        return (alpha_t * focal_weight * bce).mean()

print("✅ FocalBCELoss defined.")


✅ FocalBCELoss defined.


In [7]:
import pickle, glob, zipfile

args.ocr_cache_dir = "/kaggle/working/ocr_cache"
Path(args.ocr_cache_dir).mkdir(parents=True, exist_ok=True)

OCR_SAVE_PATH = Path("/kaggle/working/ocr_cache/ocr_full_dataset.pkl")
CHUNK_GLOB    = "/kaggle/input/datasets/arunjatothu/ocr-chunk/ocr_chunk_*.pkl"

if OCR_SAVE_PATH.exists():
    print(f"📂 Consolidated OCR file already exists, loading...")
    with open(OCR_SAVE_PATH, "rb") as f:
        ocr_data = pickle.load(f)
    print(f"✅ Loaded {len(ocr_data)} entries  ({OCR_SAVE_PATH.stat().st_size/1e6:.1f} MB)")
else:
    chunk_files = sorted(glob.glob(CHUNK_GLOB))
    zip_files = sorted(glob.glob("/kaggle/input/datasets/arunjatothu/ocr-chunk/*.zip"))
    if not chunk_files and zip_files:
        print(f"📦 Extracting {len(zip_files)} zip file(s)...")
        extract_dir = "/kaggle/working/ocr_chunks_extracted"
        os.makedirs(extract_dir, exist_ok=True)
        for zf_path in tqdm(zip_files, desc="Unzipping"):
            with zipfile.ZipFile(zf_path, 'r') as zf:
                zf.extractall(extract_dir)
        chunk_files = sorted(glob.glob(f"{extract_dir}/ocr_chunk_*.pkl"))

    if not chunk_files:
        raise FileNotFoundError(
            f"No chunk files found at {CHUNK_GLOB}\n"
            "Run the chunk notebooks first, or set CHUNK_GLOB to the right path.")

    print(f"🔨 Merging {len(chunk_files)} chunk files...")
    ocr_data = {}
    for chunk_path in tqdm(chunk_files, desc="Merging chunks"):
        with open(chunk_path, "rb") as f:
            chunk = pickle.load(f)
        for k, v in chunk.items():
            clean_key = str(k).replace("img/", "").replace("img\\", "").split("/")[-1]
            ocr_data[clean_key] = v
    with open(OCR_SAVE_PATH, "wb") as f:
        pickle.dump(ocr_data, f)
    print(f"✅ Merged OCR dataset saved → {OCR_SAVE_PATH}  ({len(ocr_data)} entries)")

def _get_wb(entry):
    if isinstance(entry, dict):
        return entry["words"], entry["boxes"]
    return entry[0], entry[1]

ocr_lookup = {fname: _get_wb(v) for fname, v in ocr_data.items()}
print(f"✅ ocr_lookup ready — {len(ocr_lookup)} entries")


📂 Consolidated OCR file already exists, loading...
✅ Loaded 12140 entries  (3.0 MB)
✅ ocr_lookup ready — 12140 entries


In [8]:
DATA_ROOT = Path(args.dataset_root) / 'hateful_memes'
IMG_ROOT  = DATA_ROOT / 'img'
LABEL_DIR = Path('/kaggle/working/labels')
LABEL_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = {
    'train':       DATA_ROOT / 'train.jsonl',
    'dev_seen':    DATA_ROOT / 'dev_seen.jsonl',
    'dev_unseen':  DATA_ROOT / 'dev_unseen.jsonl',
    'test_seen':   DATA_ROOT / 'test_seen.jsonl',
    'test_unseen': DATA_ROOT / 'test_unseen.jsonl',
}

dfs = []
for split_name, jsonl_path in SPLITS.items():
    if not jsonl_path.exists():
        print(f"⚠️  Missing: {jsonl_path}")
        continue
    rows = [json.loads(l) for l in open(jsonl_path)]
    df   = pd.DataFrame(rows)
    df['split'] = split_name
    if 'label' not in df.columns:
        df['label'] = -1
    dfs.append(df)
    print(f"   {split_name:15s}: {len(df):5d} samples | hateful: {(df.label==1).sum()}")

full_df = pd.concat(dfs, ignore_index=True)
if 'img' in full_df.columns and 'image' not in full_df.columns:
    full_df['image'] = full_df['img'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x)

csv_path = LABEL_DIR / 'hmc_info.csv'
full_df.to_csv(csv_path, index=False)
print(f"\n✅ Merged CSV saved → {csv_path}  ({len(full_df)} rows)")


   train          :  8500 samples | hateful: 3019
   dev_seen       :   500 samples | hateful: 247
   dev_unseen     :   540 samples | hateful: 200
   test_seen      :  1000 samples | hateful: 490
   test_unseen    :  2000 samples | hateful: 750

✅ Merged CSV saved → /kaggle/working/labels/hmc_info.csv  (12540 rows)


In [9]:
# ═══════════════════════════════════════════════════════
# Combiner  (unchanged from original ISSUES)
# ═══════════════════════════════════════════════════════

class Combiner(nn.Module):
    def __init__(self, convex_tensor, input_dim, comb_proj, comb_fusion):
        super().__init__()
        self.map_dim       = input_dim
        self.comb_proj     = comb_proj
        self.comb_fusion   = comb_fusion
        self.convex_tensor = convex_tensor

        branch_out_dim      = input_dim if convex_tensor else 1
        comb_in_dim         = input_dim
        comb_concat_out_dim = comb_in_dim

        if comb_proj:
            self.comb_image_proj = nn.Sequential(
                nn.Linear(comb_in_dim, 2*comb_in_dim), nn.ReLU(), nn.Dropout(0.5))
            self.comb_text_proj  = nn.Sequential(
                nn.Linear(comb_in_dim, 2*comb_in_dim), nn.ReLU(), nn.Dropout(0.5))
            comb_in_dim = 2 * comb_in_dim

        branch_in_dim = (2 * comb_in_dim) if comb_fusion == 'concat' else comb_in_dim

        self.comb_shared_branch = nn.Sequential(
            nn.Linear(branch_in_dim, 2*branch_in_dim), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(2*branch_in_dim, branch_out_dim), nn.Sigmoid())
        self.comb_concat_branch = nn.Sequential(
            nn.Linear(branch_in_dim, 2*branch_in_dim), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(2*branch_in_dim, comb_concat_out_dim))

    def forward(self, img_projection, post_features):
        pi = self.comb_image_proj(img_projection) if self.comb_proj else img_projection
        pt = self.comb_text_proj(post_features)   if self.comb_proj else post_features
        cf = torch.cat([pi, pt], dim=1) if self.comb_fusion == 'concat' else torch.mul(pi, pt)
        side    = self.comb_shared_branch(cf)
        central = self.comb_concat_branch(cf)
        return central + ((1 - side) * img_projection + side * post_features)

print("✅ Combiner defined.")


✅ Combiner defined.


In [10]:
# ═══════════════════════════════════════════════════════
# TextualInversion  (Phi weights frozen — used as-is)
# ═══════════════════════════════════════════════════════

PHI_INPUT_DIM = 768

class TextualInversion(nn.Module):
    def __init__(self, clip_model, clip_img_enc_output_dim,
                 phi_proj, text_proj, post_proj,
                 drop_probs, phi_freeze, enh_text,
                 post_dim=None, num_pre_proj_layers=1):
        super().__init__()
        self.clip_model = clip_model
        self.phi_proj   = phi_proj
        self.text_proj  = text_proj
        self.post_proj  = post_proj
        self.enh_text   = enh_text
        self.output_dim = post_dim if post_proj else clip_model.token_embedding.embedding_dim

        self.phi = nn.Sequential(
            nn.Linear(PHI_INPUT_DIM, 3072), nn.GELU(), nn.Dropout(0.5),
            nn.Linear(3072, 3072),          nn.GELU(), nn.Dropout(0.5),
            nn.Linear(3072, PHI_INPUT_DIM))

        if phi_proj:
            self.phi_map = nn.Sequential(
                nn.Linear(PHI_INPUT_DIM, PHI_INPUT_DIM), nn.Dropout(drop_probs[0]))

        phi_dict = torch.load(args.phi_weights_path, map_location='cpu')["MLPCustom"]
        with torch.no_grad():
            for i, layer_i in enumerate([0, 3, 6]):
                self.phi[layer_i].weight.copy_(phi_dict[f'layers.{layer_i}.weight'])
                self.phi[layer_i].bias.copy_(phi_dict[f'layers.{layer_i}.bias'])

        for p in self.phi.parameters():
            p.requires_grad_(False)

        layers = [nn.Linear(clip_img_enc_output_dim, PHI_INPUT_DIM), nn.Dropout(drop_probs[0])]
        for _ in range(1, num_pre_proj_layers):
            layers += [nn.ReLU(), nn.Linear(PHI_INPUT_DIM, PHI_INPUT_DIM), nn.Dropout(drop_probs[0])]
        self.pre_inversion_map = nn.Sequential(*layers)

        if post_proj:
            self.post_inversion_map = nn.Sequential(
                nn.Linear(clip_model.token_embedding.embedding_dim, post_dim),
                nn.Dropout(drop_probs[0]))

    def encode_with_vstar(self, clip_model, text, v_star, num_vstar=1,
                          pooling=True, token_id=259, proj=True):
        x = clip_model.token_embedding(text).type(clip_model.dtype)
        _, counts = torch.unique((text == token_id).nonzero(as_tuple=True)[0], return_counts=True)
        cum_sum = torch.cat((torch.zeros(1, device=x.device).int(),
                             torch.cumsum(counts, dim=0)[:-1]))
        first_vstar_indexes = (text == token_id).nonzero()[cum_sum][:, 1]
        rep_idx = torch.cat([(first_vstar_indexes + n).unsqueeze(0) for n in range(num_vstar)])
        if v_star.shape[0] == x.shape[0]:
            if len(v_star.shape) == 2: v_star = v_star.unsqueeze(1)
            x[torch.arange(x.shape[0]).repeat_interleave(num_vstar).reshape(x.shape[0], num_vstar),
              rep_idx.T] = v_star.to(x.dtype)
        x = x + clip_model.positional_embedding.type(clip_model.dtype)
        x = clip_model.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)
        x = clip_model.ln_final(x).type(clip_model.dtype)
        if pooling:
            x = (x[torch.arange(x.shape[0]), text.argmax(dim=-1)] @ clip_model.text_projection
                 if proj else x[torch.arange(x.shape[0]), text.argmax(dim=-1)])
        return x

    def forward(self, prompt, image_features):
        img_f  = self.pre_inversion_map(image_features)
        v_star = self.phi(img_f)
        if self.phi_proj: v_star = self.phi_map(v_star)
        features = self.encode_with_vstar(
            self.clip_model, prompt, v_star, proj=self.text_proj).float()
        if self.post_proj: features = self.post_inversion_map(features)
        return features

print("✅ TextualInversion defined (Phi weights frozen).")


✅ TextualInversion defined (Phi weights frozen).


In [11]:
# ═══════════════════════════════════════════════════════
# LiLTLayoutBranch — replaces LayoutLMv3 LayoutBranch
#
# Key differences vs LayoutLMv3:
#   • NO pixel_values input — eliminates redundant vision pathway
#   • RoBERTa text encoder side: FULLY FROZEN
#   • Only layout Transformer attention weights get LoRA (r=4)
#   • ~70% fewer effective trainable params in the layout branch
# ═══════════════════════════════════════════════════════

LILT_HIDDEN = 768

class LiLTLayoutBranch(nn.Module):
    """
    LiLT layout branch with LoRA on layout attention only.
    Text encoder (RoBERTa side) is completely frozen.
    Takes: input_ids, attention_mask, bbox  (no pixel_values)
    Returns: (B, output_dim) layout embedding
    """
    def __init__(self, model_name: str, output_dim: int, dropout: float,
                 lora_r: int, lora_alpha: int, lora_dropout: float,
                 lora_target_modules: list):
        super().__init__()

        base_model = LiltModel.from_pretrained(model_name)

        # ── Freeze RoBERTa text side completely ───────────────────────────
        # LiLT parameter naming: text encoder params do NOT have 'layout_'
        # prefix; layout Transformer params have 'layout_' prefix.
        n_frozen = 0
        for name, param in base_model.named_parameters():
            if 'layout_' not in name:        # text encoder / embeddings
                param.requires_grad_(False)
                n_frozen += param.numel()
        print(f"   LiLT: Froze {n_frozen/1e6:.2f}M text-side params")

        # ── Apply LoRA only to layout attention projections ───────────────
        lora_cfg = LoraConfig(
            r                = lora_r,
            lora_alpha       = lora_alpha,
            target_modules   = lora_target_modules,  # layout_query, layout_value
            lora_dropout     = lora_dropout,
            bias             = "none",
            task_type        = TaskType.FEATURE_EXTRACTION,
        )
        self.encoder = get_peft_model(base_model, lora_cfg)

        n_trainable = sum(p.numel() for p in self.encoder.parameters() if p.requires_grad)
        n_total     = sum(p.numel() for p in self.encoder.parameters())
        print(f"   LiLTLayoutBranch (LoRA r={lora_r}): "
              f"{n_trainable/1e6:.3f}M / {n_total/1e6:.2f}M params trainable "
              f"({100*n_trainable/n_total:.2f}%)")

        # ── Projection head (trained from scratch) ────────────────────────
        self.proj = nn.Sequential(
            nn.Linear(LILT_HIDDEN, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, input_ids, attention_mask, bbox):
        """
        Args:
            input_ids      : (B, L)  tokenized OCR words
            attention_mask : (B, L)
            bbox           : (B, L, 4)  normalised 0-1000 bounding boxes
        Returns:
            (B, output_dim)
        """
        outputs  = self.encoder(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            bbox           = bbox,
        )
        cls_feat = outputs.last_hidden_state[:, 0, :]   # (B, 768)
        return self.proj(cls_feat)                       # (B, output_dim)

print("✅ LiLTLayoutBranch defined (RoBERTa frozen, LoRA on layout attention only).")


✅ LiLTLayoutBranch defined (RoBERTa frozen, LoRA on layout attention only).


In [12]:
class TriModalCombiner(nn.Module):
    """Fuses visual, text-inversion, and layout features. Trained from scratch."""
    def __init__(self, map_dim, layout_dim, comb_proj, comb_fusion, convex_tensor):
        super().__init__()
        self.bimodal = Combiner(convex_tensor, map_dim, comb_proj, comb_fusion)

        self.layout_up = nn.Sequential(
            nn.Linear(layout_dim, map_dim),
            nn.LayerNorm(map_dim),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        gate_in = 2 * map_dim
        self.gate = nn.Sequential(
            nn.Linear(gate_in, map_dim), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(map_dim, 1), nn.Sigmoid()
        )
        self.layout_branch_proj = nn.Sequential(
            nn.Linear(gate_in, map_dim), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(map_dim, map_dim)
        )
        self.branch_drop = nn.Dropout(p=0.05)

    def forward(self, visual_feat, text_inv_feat, layout_feat):
        bi  = self.bimodal(visual_feat, text_inv_feat)
        lf  = self.layout_up(layout_feat)
        cat = torch.cat([bi, lf], dim=1)
        g   = self.gate(cat)
        lc  = self.layout_branch_proj(cat)
        return (1 - g) * bi + g * self.branch_drop(lc)

print("✅ TriModalCombiner defined.")


✅ TriModalCombiner defined.


In [13]:
class MemesDataset(Dataset):
    def __init__(self, data_root, split='train', image_size=224, fast=False, ocr_lookup=None):
        super().__init__()
        self.data_root  = Path(data_root) / 'hateful_memes'
        self.img_root   = self.data_root / 'img'
        self.split      = split
        self.image_size = image_size
        self.ocr_lookup = ocr_lookup or {}

        csv_path = Path('/kaggle/working/labels/hmc_info.csv')
        self.df  = pd.read_csv(csv_path)
        self.df  = self.df[self.df['split'] == split].reset_index(drop=True)
        float_cols = self.df.select_dtypes(float).columns
        self.df[float_cols] = self.df[float_cols].fillna(-1).astype('Int64')

        if self.ocr_lookup:
            missing = [str(fn).split('/')[-1]
                       for fn in self.df['image']
                       if str(fn).split('/')[-1] not in self.ocr_lookup]
            if missing:
                print(f"  ⚠️  [{split}] {len(missing)} images not in ocr_lookup — [UNK] fallback")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        txt    = 'null' if str(row.get('text', '')).lower() == 'nothing' else str(row.get('text', ''))
        img_fn = row.get('image', row.get('img', ''))
        if isinstance(img_fn, str) and '/' in img_fn:
            img_fn = img_fn.split('/')[-1]
        image  = Image.open(self.img_root / img_fn).convert('RGB').resize((self.image_size, self.image_size))
        words, boxes = self.ocr_lookup.get(str(img_fn), (['[UNK]'], [[0, 0, 1000, 1000]]))
        return {
            'image':       image,
            'text':        txt,
            'label':       int(row.get('label', -1)),
            'idx_meme':    int(row.get('id', idx)),
            'origin_text': txt,
            'img_path':    str(self.img_root / img_fn),
            'ocr_words':   words,
            'ocr_boxes':   boxes,
            'is_train':    self.split == 'train',
        }


# ── LiLT Collator ─────────────────────────────────────────────────────────
# Key difference: uses RobertaTokenizerFast directly — no LayoutLMv3Processor.
# Bounding boxes are passed separately and aligned to tokenized subwords.
# NO pixel_values for the layout branch.

class MemesCollator:
    def __init__(self, args):
        self.args = args
        _, self.clip_preprocess = clip.load("ViT-L/14", device='cpu', jit=False)
        self._lilt_tokenizer = None
        self.train_aug = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.1),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
            T.RandomGrayscale(p=0.08),
            T.RandomRotation(degrees=10),
        ])

    @property
    def lilt_tokenizer(self):
        if self._lilt_tokenizer is None:
            # LiLT (lilt-roberta-en-base) uses RoBERTa tokenizer
            self._lilt_tokenizer = RobertaTokenizerFast.from_pretrained(
                args.lilt_model, add_prefix_space=True)
        return self._lilt_tokenizer

    def _encode_lilt(self, batch_words, batch_boxes, max_length=512):
        """
        Tokenize word-level OCR output and manually align bounding boxes to subwords.
        RobertaTokenizerFast does not accept a 'boxes' arg — alignment is done manually.
        """
        tok = self.lilt_tokenizer
    
        all_input_ids      = []
        all_attention_mask = []
        all_bbox           = []
    
        for words, boxes in zip(batch_words, batch_boxes):
            if not words or words == ['[UNK]']:
                words = ['meme']
                boxes = [[0, 0, 1000, 1000]]
    
            # Clamp boxes to [0, 1000] — LiLT expects normalised coords
            boxes = [[max(0, min(1000, c)) for c in b] for b in boxes]
    
            enc = tok(
                words,
                is_split_into_words = True,
                max_length          = max_length,
                padding             = 'max_length',
                truncation          = True,
                return_tensors      = 'pt',
            )
    
            # Align box to each subword token using word_ids()
            word_ids    = enc.word_ids(batch_index=0)   # list of word idx per token, None = special
            aligned_box = []
            for wid in word_ids:
                if wid is None:
                    aligned_box.append([0, 0, 0, 0])   # [CLS], [SEP], [PAD]
                else:
                    aligned_box.append(boxes[wid])
    
            all_input_ids.append(enc['input_ids'])
            all_attention_mask.append(enc['attention_mask'])
            all_bbox.append(torch.tensor([aligned_box], dtype=torch.long))  # (1, L, 4)
    
        return {
            'input_ids':      torch.cat(all_input_ids,      dim=0),   # (B, L)
            'attention_mask': torch.cat(all_attention_mask, dim=0),   # (B, L)
            'bbox':           torch.cat(all_bbox,           dim=0),   # (B, L, 4)
        }

    def __call__(self, batch):
        labels    = torch.LongTensor([item['label']    for item in batch])
        idx_memes = torch.LongTensor([item['idx_meme'] for item in batch])
        img_paths = [item['img_path'] for item in batch]

        enh_texts = torch.cat([
            clip.tokenize(f'a photo of $ , {el["origin_text"]}',
                          context_length=77, truncate=True)
            for el in batch], dim=0)
        simple_prompt = clip.tokenize(
            'a photo of $', context_length=77).repeat(labels.shape[0], 1)

        img = torch.cat([
            self.clip_preprocess(
                self.train_aug(item['image']) if item.get('is_train') else item['image']
            ).unsqueeze(0)
            for item in batch], dim=0)

        texts = torch.cat([clip.tokenize(item['text'], context_length=77, truncate=True)
                           for item in batch], dim=0)

        out = {
            'labels': labels, 'idx_memes': idx_memes,
            'enhanced_texts': enh_texts, 'simple_prompt': simple_prompt,
            'pixel_values': img, 'texts': texts, 'img_paths': img_paths,
        }

        if args.name == 'issues-layout':
            # LiLT encoding — no pixel_values
            lilt_enc = self._encode_lilt(
                [item['ocr_words'] for item in batch],
                [item['ocr_boxes'] for item in batch],
                max_length=args.lilt_max_length,
            )
            out['layout_input_ids']      = lilt_enc['input_ids']
            out['layout_attention_mask'] = lilt_enc['attention_mask']
            out['layout_bbox']           = lilt_enc['bbox']
            # NOTE: no layout_pixel_values — LiLT does not use vision backbone

        return out

print("✅ MemesDataset & LiLT MemesCollator defined.")


✅ MemesDataset & LiLT MemesCollator defined.


In [14]:
class EpochLogger(pl.Callback):
    COLS = [
        ('epoch',      'Ep',       5),
        ('train_loss', 'TrLoss',   8),
        ('val_loss',   'VaLoss',   8),
        ('train_acc',  'TrAcc',    7),
        ('val_acc',    'VaAcc',    7),
        ('train_f1',   'TrF1',     7),
        ('val_f1',     'VaF1',     7),
        ('train_auroc','TrAUROC',  9),
        ('val_auroc',  'VaAUROC',  9),
    ]

    def __init__(self, log_path: str, experiment_name: str):
        super().__init__()
        self.log_path        = log_path
        self.experiment_name = experiment_name
        self.records         = []

    @staticmethod
    def _f(val): return f"{val:.4f}" if val == val else "  —   "

    def _header(self):
        parts = [f"{h:^{w}}" for _, h, w in self.COLS]
        sep   = ["-"*w       for _, _, w in self.COLS]
        return "  |  ".join(parts) + "\n" + "--+--".join(sep)

    def _row(self, rec, best_epoch):
        star  = "★" if int(rec["epoch"]) == best_epoch else " "
        parts = []
        for key, _, w in self.COLS:
            val = rec.get(key, float("nan"))
            s   = (f"{int(val):^{w}}" if key == "epoch"
                   else f"{self._f(val):^{w}}")
            parts.append(s)
        return star + " " + "  |  ".join(parts)

    def _print_table(self):
        if not self.records: return
        df      = pd.DataFrame(self.records)
        best_ep = int(df.loc[df["val_auroc"].idxmax(), "epoch"])
        sep     = "=" * (sum(w for _, _, w in self.COLS) + 5*(len(self.COLS)-1) + 2)
        tqdm.write("")
        tqdm.write(f"  📋  Epoch History — {self.experiment_name}")
        tqdm.write(sep)
        tqdm.write("  " + self._header())
        tqdm.write(sep)
        for _, row in df.iterrows():
            tqdm.write(self._row(row.to_dict(), best_ep))
        tqdm.write(sep)
        tqdm.write(f"  ★ = best val AUROC (epoch {best_ep})")
        tqdm.write("")

    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking: return
        m = trainer.callback_metrics
        record = {
            "epoch":      trainer.current_epoch + 1,
            "train_loss": float(m.get("train/loss",      float("nan"))),
            "val_loss":   float(m.get("val/loss",        float("nan"))),
            "train_acc":  float(m.get("train/accuracy",  float("nan"))),
            "val_acc":    float(m.get("val/accuracy",    float("nan"))),
            "train_f1":   float(m.get("train/f1",        float("nan"))),
            "val_f1":     float(m.get("val/f1",          float("nan"))),
            "train_auroc":float(m.get("train/auroc",     float("nan"))),
            "val_auroc":  float(m.get("val/auroc",       float("nan"))),
        }
        self.records.append(record)
        pd.DataFrame(self.records).to_csv(self.log_path, index=False)
        self._print_table()

print("✅ EpochLogger defined.")


✅ EpochLogger defined.


In [15]:
CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ = 1024

class LinearProjection(nn.Module):
    def __init__(self, input_dim, output_dim, num_layers, drop_probs):
        super().__init__()
        layers = [nn.Linear(input_dim, output_dim), nn.Dropout(drop_probs[0])]
        for _ in range(1, num_layers):
            layers += [nn.ReLU(), nn.Linear(output_dim, output_dim), nn.Dropout(drop_probs[0])]
        self.proj = nn.Sequential(*layers)
    def forward(self, x): return self.proj(x)


class HateClassifier(pl.LightningModule):
    """
    ISSUES + LiLT variant.
    Frozen: CLIP backbone, Phi weights, RoBERTa text side of LiLT
    Trainable: LoRA adapters on LiLT layout attention, all projections & combiners
    """

    def __init__(self, args):
        super().__init__()
        self.args               = args
        self.dataset            = args.dataset
        self.num_mapping_layers = args.num_mapping_layers
        self.map_dim            = args.map_dim
        self.fusion             = args.fusion
        self.num_pre_output_layers = args.num_pre_output_layers
        self.lr                 = args.lr
        self.weight_decay       = args.weight_decay
        self.name               = args.name
        self.fast_process       = args.fast_process
        self.proj_map           = args.proj_map
        self.convex_tensor      = args.convex_tensor
        self.comb_proj          = args.comb_proj
        self.comb_fusion        = args.comb_fusion
        self.enh_text           = args.enh_text
        self.phi_freeze         = True
        self.text_inv_proj      = args.text_inv_proj
        self.phi_inv_proj       = args.phi_inv_proj
        self.post_inv_proj      = args.post_inv_proj
        self.label_smoothing    = args.label_smoothing

        for prefix in ('train', 'val', 'test'):
            setattr(self, f'{prefix}_acc',   torchmetrics.Accuracy(task='binary'))
            setattr(self, f'{prefix}_auroc', torchmetrics.AUROC(task='binary'))
            setattr(self, f'{prefix}_prec',  torchmetrics.Precision(task='binary'))
            setattr(self, f'{prefix}_rec',   torchmetrics.Recall(task='binary'))
            setattr(self, f'{prefix}_f1',    torchmetrics.F1Score(task='binary'))

        # ── CLIP backbone (frozen) ─────────────────────────────────────────
        self.clip_model, _ = clip.load("ViT-L/14", device='cpu', jit=False)
        self.clip_model.visual.proj = None
        self.clip_model.float()
        for p in self.clip_model.parameters():
            p.requires_grad_(False)

        self.image_map = LinearProjection(CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ,
                                          self.map_dim, self.num_mapping_layers, args.drop_probs)
        self.text_map  = LinearProjection(self.clip_model.token_embedding.embedding_dim,
                                          self.map_dim, self.num_mapping_layers, args.drop_probs)

        pre_output_input_dim = self._build_head(args)

        proj_path = getattr(args, 'proj_weights_path', None)
        if proj_path and os.path.exists(proj_path):
            proj_sd = torch.load(proj_path, map_location='cpu', weights_only=False)['state_dict']
            with torch.no_grad():
                self.image_map.proj[0].weight.copy_(proj_sd['image_proj_weight'])
                self.image_map.proj[0].bias.copy_(proj_sd['image_proj_bias'])
                self.text_map.proj[0].weight.copy_(proj_sd['text_proj_weight'])
                self.text_map.proj[0].bias.copy_(proj_sd['text_proj_bias'])
            print("✅ Loaded pre-trained ISSUES projection weights into image_map & text_map")
        else:
            print("⚠️  proj_weights_path not set or file missing — image_map & text_map initialised randomly")
        
        pre_output_layers = [nn.Dropout(args.drop_probs[1])]
        output_input_dim  = pre_output_input_dim
        if self.num_pre_output_layers >= 1:
            pre_output_layers += [nn.Linear(pre_output_input_dim, self.map_dim),
                                  nn.ReLU(), nn.Dropout(args.drop_probs[2])]
            output_input_dim = self.map_dim
        for _ in range(1, self.num_pre_output_layers):
            pre_output_layers += [nn.Linear(self.map_dim, self.map_dim),
                                  nn.ReLU(), nn.Dropout(args.drop_probs[2])]

        self.pre_output = nn.Sequential(*pre_output_layers)
        self.output     = nn.Linear(output_input_dim, 1)
        self.criterion  = nn.BCEWithLogitsLoss(
            pos_weight = torch.tensor([args.pos_weight]),
            reduction  = 'mean',
        )

    def _build_head(self, args):
        if self.name == 'issues-layout':
            self.text_inv = TextualInversion(
                self.clip_model, CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ,
                args.phi_inv_proj, args.text_inv_proj, args.post_inv_proj,
                args.drop_probs, phi_freeze=True, enh_text=args.enh_text,
                post_dim=self.map_dim, num_pre_proj_layers=args.num_mapping_layers)
            self.image_map = LinearProjection(CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ,
                                              self.map_dim, args.num_mapping_layers, args.drop_probs)
            # ── LiLT layout branch replaces LayoutLMv3 ────────────────────
            self.layout_branch = LiLTLayoutBranch(
                model_name         = args.lilt_model,
                output_dim         = args.layout_dim,
                dropout            = args.layout_dropout,
                lora_r             = args.lora_r,
                lora_alpha         = args.lora_alpha,
                lora_dropout       = args.lora_dropout,
                lora_target_modules= args.lora_target_modules,
            )
            self.tri_combiner = TriModalCombiner(
                map_dim      = self.map_dim,
                layout_dim   = args.layout_dim,
                comb_proj    = args.comb_proj,
                comb_fusion  = args.comb_fusion,
                convex_tensor= args.convex_tensor,
            )
            return self.map_dim

        elif self.name in ['hate-clipper', 'adaptation']:
            return self.map_dim if args.fusion == 'align' else self.map_dim * 2
        elif self.name in ['text-only', 'image-only']:
            return (self.map_dim if self.proj_map
                    else (self.clip_model.token_embedding.embedding_dim
                          if self.name == 'text-only' else CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ))
        elif self.name == 'sum':
            return self.map_dim
        elif self.name == 'combiner':
            self.comb = Combiner(self.convex_tensor, self.map_dim, self.comb_proj, self.comb_fusion)
            return self.map_dim
        elif self.name in ('text-inv', 'text-inv-comb'):
            self.text_inv = TextualInversion(
                self.clip_model, CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ,
                self.phi_inv_proj, self.text_inv_proj, self.post_inv_proj,
                args.drop_probs, phi_freeze=True, enh_text=self.enh_text,
                post_dim=self.map_dim, num_pre_proj_layers=self.num_mapping_layers)
            if self.name == 'text-inv-comb':
                self.comb = Combiner(self.convex_tensor, self.map_dim, self.comb_proj, self.comb_fusion)
                self.image_map = LinearProjection(CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ,
                                                  self.map_dim, self.num_mapping_layers, args.drop_probs)
            return self.text_inv.output_dim if self.name == 'text-inv' else self.map_dim
        raise ValueError(f'Unknown model name: {self.name}')

    def compute_CLIP_features(self, img, text):
        img_f = self.clip_model.visual(img.type(self.clip_model.dtype))
        x = self.clip_model.token_embedding(text).type(self.clip_model.dtype)
        x = x + self.clip_model.positional_embedding.type(self.clip_model.dtype)
        x = self.clip_model.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)
        x = self.clip_model.ln_final(x).type(self.clip_model.dtype)
        tf = x[torch.arange(x.shape[0]), text.argmax(dim=-1)]
        return img_f, tf

    # def _smooth_bce(self, logits, labels):
    #     return self.criterion(logits, labels.float())

    def _smooth_bce(self, logits, labels):
        eps    = self.label_smoothing
        labels = labels.float() * (1 - eps) + 0.5 * eps
        return self.criterion(logits, labels)

    def common_step(self, batch):
        img_f, txt_f = self.compute_CLIP_features(batch['pixel_values'], batch['texts'])
        prompt = batch['enhanced_texts'] if self.enh_text else batch['simple_prompt']
        n = self.name

        if n == 'issues-layout':
            ti_feat  = self.text_inv(prompt, img_f)
            vis_feat = self.image_map(img_f)
            # LiLT forward — no pixel_values argument
            lay_feat = self.layout_branch(
                input_ids      = batch['layout_input_ids'].to(img_f.device),
                attention_mask = batch['layout_attention_mask'].to(img_f.device),
                bbox           = batch['layout_bbox'].to(img_f.device),
            )
            features = self.tri_combiner(vis_feat, ti_feat, lay_feat)

        elif n in ['hate-clipper', 'adaptation']:
            img_f = self.image_map(img_f); txt_f = self.text_map(txt_f)
            features = torch.mul(img_f, txt_f) if self.fusion == 'align' else torch.cat([img_f, txt_f], dim=1)
        elif n == 'text-only':
            features = self.text_map(txt_f) if self.proj_map else txt_f
        elif n == 'image-only':
            features = self.image_map(img_f) if self.proj_map else img_f
        elif n == 'sum':
            features = self.image_map(img_f) + self.text_map(txt_f)
        elif n == 'combiner':
            features = self.comb(self.image_map(img_f), self.text_map(txt_f))
        elif n == 'text-inv':
            features = self.text_inv(prompt, img_f)
        elif n == 'text-inv-comb':
            features = self.comb(self.image_map(img_f), self.text_inv(prompt, img_f))
        else:
            raise ValueError(f'Unknown model name: {n}')

        logits      = self.output(self.pre_output(features)).squeeze(1)
        preds_proxy = torch.sigmoid(logits)
        preds       = (preds_proxy >= 0.5).long()
        return {
            'loss':        self._smooth_bce(logits, batch['labels'].float()),
            'preds':       preds,
            'preds_proxy': preds_proxy,
            'labels':      batch['labels'],
        }

    # ── Training ──────────────────────────────────────────────────────────
    def training_step(self, batch, batch_idx):
        out = self.common_step(batch)
        self.train_acc.update(out['preds'],         out['labels'])
        self.train_auroc.update(out['preds_proxy'], out['labels'])
        self.train_prec.update(out['preds'],        out['labels'])
        self.train_rec.update(out['preds'],         out['labels'])
        self.train_f1.update(out['preds'],          out['labels'])
        self.log('train/loss', out['loss'], on_step=True, on_epoch=True, prog_bar=True)
        self.log('train/acc',  self.train_acc.compute(),  on_step=True, on_epoch=False, prog_bar=True)
        self.log('train/f1',   self.train_f1.compute(),   on_step=True, on_epoch=False, prog_bar=True)
        return out['loss']

    def on_train_epoch_end(self):
        self.log('train/accuracy',  self.train_acc.compute(),   prog_bar=True)
        self.log('train/auroc',     self.train_auroc.compute(), prog_bar=True)
        self.log('train/precision', self.train_prec.compute())
        self.log('train/recall',    self.train_rec.compute())
        self.log('train/f1',        self.train_f1.compute(),    prog_bar=True)
        for m in (self.train_acc, self.train_auroc, self.train_prec, self.train_rec, self.train_f1):
            m.reset()

    # ── Validation ────────────────────────────────────────────────────────
    def validation_step(self, batch, batch_idx):
        out = self.common_step(batch)
        self.val_acc.update(out['preds'],         out['labels'])
        self.val_auroc.update(out['preds_proxy'], out['labels'])
        self.val_prec.update(out['preds'],        out['labels'])
        self.val_rec.update(out['preds'],         out['labels'])
        self.val_f1.update(out['preds'],          out['labels'])
        self.log('val/loss', out['loss'], on_step=False, on_epoch=True, prog_bar=True)
        self.log('val/acc',  self.val_acc.compute(), prog_bar=True)
        self.log('val/f1',   self.val_f1.compute(),  prog_bar=True)
        return out

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking: return
        self.log('val/accuracy',  self.val_acc.compute(),   prog_bar=True)
        self.log('val/auroc',     self.val_auroc.compute(), prog_bar=True)
        self.log('val/precision', self.val_prec.compute())
        self.log('val/recall',    self.val_rec.compute())
        self.log('val/f1',        self.val_f1.compute(),    prog_bar=True)
        for m in (self.val_acc, self.val_auroc, self.val_prec, self.val_rec, self.val_f1):
            m.reset()

    # ── Test ──────────────────────────────────────────────────────────────
    def test_step(self, batch, batch_idx, dataloader_idx=0):
        prefix = {0: 'dev_seen', 1: 'test_seen', 2: 'dev_unseen', 3: 'test_unseen'}.get(dataloader_idx, 'test')
        out = self.common_step(batch)
        self.test_acc.update(out['preds'],         out['labels'])
        self.test_auroc.update(out['preds_proxy'], out['labels'])
        self.log(f'{prefix}/accuracy', self.test_acc.compute())
        self.log(f'{prefix}/auroc',    self.test_auroc.compute())
        return out

    # ── Optimiser — two param groups: LoRA (lower LR) vs scratch ──────────
    def configure_optimizers(self):
        lora_params   = []
        other_params  = []
        no_decay      = ['bias', 'LayerNorm.weight', 'layer_norm.weight']
    
        for name, p in self.named_parameters():
            if not p.requires_grad:
                continue
            if 'layout_branch.encoder' in name:
                lora_params.append(p)
            else:
                other_params.append((name, p))
    
        param_groups = [
            # scratch params with weight decay
            {'params': [p for n, p in other_params if not any(nd in n for nd in no_decay)],
             'lr': self.lr, 'weight_decay': self.weight_decay, 'name': 'scratch_decay'},
            # scratch params without weight decay (biases, norms)
            {'params': [p for n, p in other_params if any(nd in n for nd in no_decay)],
             'lr': self.lr, 'weight_decay': 0.0, 'name': 'scratch_nodecay'},
            # LoRA — low LR, no weight decay
            {'params': lora_params, 'lr': self.lr * 0.1,
             'weight_decay': 0.0, 'name': 'lora'},
        ]
        param_groups = [g for g in param_groups if g['params']]
        optimizer = torch.optim.AdamW(param_groups)
    
        from transformers import get_cosine_schedule_with_warmup
        train_loader    = self.trainer.fit_loop._data_source.dataloader()
        steps_per_epoch = len(train_loader) // max(1, self.trainer.accumulate_grad_batches)
        total_steps     = steps_per_epoch * self.args.max_epochs
        warmup_steps    = steps_per_epoch * 2   # 2 epoch warmup for stability
    
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = warmup_steps,
            num_training_steps = total_steps,
        )
        return {
            'optimizer':    optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'interval': 'step', 'frequency': 1},
        }

print("✅ HateClassifier (LiLT + from-scratch projection) defined.")


✅ HateClassifier (LiLT + from-scratch projection) defined.


## 🚀 Build Dataloaders & Start Training

In [16]:
seed_everything(42, workers=True)
collator = MemesCollator(args)

ds_train       = MemesDataset(args.dataset_root, 'train',       args.image_size, ocr_lookup=ocr_lookup)
ds_val         = MemesDataset(args.dataset_root, 'dev_seen',    args.image_size, ocr_lookup=ocr_lookup)
ds_test        = MemesDataset(args.dataset_root, 'test_seen',   args.image_size, ocr_lookup=ocr_lookup)
ds_val_unseen  = MemesDataset(args.dataset_root, 'dev_unseen',  args.image_size, ocr_lookup=ocr_lookup)
ds_test_unseen = MemesDataset(args.dataset_root, 'test_unseen', args.image_size, ocr_lookup=ocr_lookup)

NUM_WORKERS = 2

dl_train = DataLoader(ds_train, batch_size=args.batch_size, shuffle=True,
                      collate_fn=collator, num_workers=NUM_WORKERS,
                      pin_memory=True, persistent_workers=True, prefetch_factor=2)
dl_val        = DataLoader(ds_val,         batch_size=args.batch_size, shuffle=False,
                           collate_fn=collator, num_workers=NUM_WORKERS,
                           persistent_workers=True, prefetch_factor=2)
dl_test       = DataLoader(ds_test,        batch_size=args.batch_size, shuffle=False,
                           collate_fn=collator, num_workers=NUM_WORKERS,
                           persistent_workers=True, prefetch_factor=2)
dl_val_unseen = DataLoader(ds_val_unseen,  batch_size=args.batch_size, shuffle=False,
                           collate_fn=collator, num_workers=NUM_WORKERS,
                           persistent_workers=True, prefetch_factor=2)
dl_test_unseen= DataLoader(ds_test_unseen, batch_size=args.batch_size, shuffle=False,
                           collate_fn=collator, num_workers=NUM_WORKERS,
                           persistent_workers=True, prefetch_factor=2)

print(f"Train:{len(ds_train):,}  Val:{len(ds_val):,}  Test:{len(ds_test):,}  "
      f"Val-Unseen:{len(ds_val_unseen):,}  Test-Unseen:{len(ds_test_unseen):,}")


Train:8,500  Val:500  Test:1,000  Val-Unseen:540  Test-Unseen:2,000


In [17]:
run_name     = f"{args.name}_{args.dataset}_{args.experiment_name}"
log_csv_path = f"{args.experiment_log_dir}/{run_name}_epoch_log.csv"

epoch_logger = EpochLogger(log_csv_path, run_name)

ckpt_callback = ModelCheckpoint(
    dirpath          = args.checkpoint_dir,
    filename         = run_name + '-{epoch:02d}-{val/auroc:.4f}',
    monitor          = 'val/auroc',
    mode             = 'max',
    save_top_k       = 2,
    save_last        = True,
    save_weights_only= True,
    verbose          = True,
)
early_stop = EarlyStopping(
    monitor  = 'val/auroc',   # AUROC is smoother than F1 for binary
    patience = 7,
    mode     = 'max',         # was 'min'
    verbose  = True,
)

model = HateClassifier(args)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Parameter summary:")
print(f"   Total      : {total/1e6:.1f}M")
print(f"   Trainable  : {trainable/1e6:.1f}M")
print(f"   Frozen     : {(total-trainable)/1e6:.1f}M")
print(f"   (CLIP + Phi + LiLT RoBERTa text side = frozen; "
      f"LiLT LoRA adapters + all projection/combiner layers = trainable)")

trainer = Trainer(
    accelerator            = 'gpu' if device == 'cuda' else 'cpu',
    devices                = 1,
    max_epochs             = args.max_epochs,
    gradient_clip_val      = args.gradient_clip_val,
    accumulate_grad_batches= 2,          # effective batch = 32, smoother gradients
    log_every_n_steps      = args.log_every_n_steps,
    val_check_interval     = args.val_check_interval,
    limit_train_batches    = args.limit_train_batches,
    limit_val_batches      = args.limit_val_batches,
    callbacks              = [ckpt_callback, early_stop, epoch_logger],
    deterministic          = False,
    enable_progress_bar    = True,
    precision              = '16-mixed',
)
print(f"\n🚀 Ready — run: {run_name}")


Loading weights:   0%|          | 0/400 [00:00<?, ?it/s]

   LiLT: Froze 124.65M text-side params


   LiLTLayoutBranch (LoRA r=2): 0.018M / 130.77M params trainable (0.01%)
✅ Loaded pre-trained ISSUES projection weights into image_map & text_map

📊 Parameter summary:
   Total      : 588.2M
   Trainable  : 16.5M
   Frozen     : 571.7M
   (CLIP + Phi + LiLT RoBERTa text side = frozen; LiLT LoRA adapters + all projection/combiner layers = trainable)

🚀 Ready — run: issues-layout_hmc_issues_lilt_lora_v1


In [18]:
trainer.fit(model, train_dataloaders=dl_train, val_dataloaders=dl_val)
print(f"\n✅ Training complete.  Best checkpoint: {ckpt_callback.best_model_path}")


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


  📋  Epoch History — issues-layout_hmc_issues_lilt_lora_v1
   Ep    |   TrLoss   |   VaLoss   |   TrAcc   |   VaAcc   |   TrF1    |   VaF1    |   TrAUROC   |   VaAUROC 
-------+------------+------------+-----------+-----------+-----------+-----------+-------------+-----------
★   1    |   0.9458   |   0.9816   |    —      |  0.5902   |  0.4516   |  0.6707   |     —       |   0.6528  
  ★ = best val AUROC (epoch 1)



Validation: |          | 0/? [00:00<?, ?it/s]


  📋  Epoch History — issues-layout_hmc_issues_lilt_lora_v1
   Ep    |   TrLoss   |   VaLoss   |   TrAcc   |   VaAcc   |   TrF1    |   VaF1    |   TrAUROC   |   VaAUROC 
-------+------------+------------+-----------+-----------+-----------+-----------+-------------+-----------
    1    |   0.9458   |   0.9816   |    —      |  0.5902   |  0.4516   |  0.6707   |     —       |   0.6528  
★   2    |   0.8907   |   0.9204   |  0.5331   |  0.6660   |  0.5613   |  0.6867   |   0.5509    |   0.7247  
  ★ = best val AUROC (epoch 2)



... (11 outputs left out)


  📋  Epoch History — issues-layout_hmc_issues_lilt_lora_v1
   Ep    |   TrLoss   |   VaLoss   |   TrAcc   |   VaAcc   |   TrF1    |   VaF1    |   TrAUROC   |   VaAUROC 
-------+------------+------------+-----------+-----------+-----------+-----------+-------------+-----------
    1    |   0.9458   |   0.9816   |    —      |  0.5902   |  0.4516   |  0.6707   |     —       |   0.6528  
    2    |   0.8907   |   0.9204   |  0.5331   |  0.6660   |  0.5613   |  0.6867   |   0.5509    |   0.7247  
    3    |   0.8506   |   0.9069   |  0.5914   |  0.6720   |  0.5912   |  0.6858   |   0.6810    |   0.7342  
    4    |   0.8148   |   0.9355   |  0.6516   |  0.6960   |  0.6271   |  0.6872   |   0.7332    |   0.7404  
    5    |   0.8017   |   0.9419   |  0.6839   |  0.6840   |  0.6427   |  0.6708   |   0.7756    |   0.7434  
    6    |   0.7776   |   0.9346   |  0.7006   |  0.6880   |  0.6580   |  0.6777   |   0.7895    |   0.7454  
    7    |   0.7651   |   0.9401   |  0.7178   |  0.6780   |  

Validation: |          | 0/? [00:00<?, ?it/s]


  📋  Epoch History — issues-layout_hmc_issues_lilt_lora_v1
   Ep    |   TrLoss   |   VaLoss   |   TrAcc   |   VaAcc   |   TrF1    |   VaF1    |   TrAUROC   |   VaAUROC 
-------+------------+------------+-----------+-----------+-----------+-----------+-------------+-----------
    1    |   0.9458   |   0.9816   |    —      |  0.5902   |  0.4516   |  0.6707   |     —       |   0.6528  
    2    |   0.8907   |   0.9204   |  0.5331   |  0.6660   |  0.5613   |  0.6867   |   0.5509    |   0.7247  
    3    |   0.8506   |   0.9069   |  0.5914   |  0.6720   |  0.5912   |  0.6858   |   0.6810    |   0.7342  
    4    |   0.8148   |   0.9355   |  0.6516   |  0.6960   |  0.6271   |  0.6872   |   0.7332    |   0.7404  
    5    |   0.8017   |   0.9419   |  0.6839   |  0.6840   |  0.6427   |  0.6708   |   0.7756    |   0.7434  
    6    |   0.7776   |   0.9346   |  0.7006   |  0.6880   |  0.6580   |  0.6777   |   0.7895    |   0.7454  
    7    |   0.7651   |   0.9401   |  0.7178   |  0.6780   |  


✅ Training complete.  Best checkpoint: /kaggle/working/checkpoints/issues-layout_hmc_issues_lilt_lora_v1-epoch=08-val/auroc=0.7533.ckpt


## 🧪 Evaluation on All Test Splits

In [19]:
best_ckpt = ckpt_callback.best_model_path
# best_ckpt = '/kaggle/working/checkpoints/...'   # override if needed

print(f"Loading best checkpoint: {best_ckpt}")
best_model = HateClassifier.load_from_checkpoint(best_ckpt, args=args)
best_model.eval().to(device)

def collect_predictions(model, dataloader, split_name):
    model.eval()
    all_preds, all_probs, all_labels, all_paths = [], [], [], []
    running_correct = 0; running_total = 0
    running_probs = []; running_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc=f"  {split_name}", ncols=100, leave=False)
        for batch in pbar:
            batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v)
                     for k, v in batch.items()}
            out   = model.common_step(batch)
            preds  = out['preds'].cpu().numpy()
            probs  = out['preds_proxy'].cpu().numpy()
            labels = out['labels'].cpu().numpy()
            all_preds.append(preds); all_probs.append(probs)
            all_labels.append(labels)
            all_paths.extend(batch.get('img_paths', [''] * len(preds)))
            valid = labels >= 0
            if valid.any():
                running_correct += (preds[valid] == labels[valid]).sum()
                running_total   += valid.sum()
                running_probs.extend(probs[valid].tolist())
                running_labels.extend(labels[valid].tolist())
            if running_total > 0:
                running_acc   = running_correct / running_total
                running_auroc = roc_auc_score(running_labels, running_probs) if len(set(running_labels)) > 1 else 0.5
            else:
                running_acc, running_auroc = 0.0, 0.5
            pbar.set_postfix({'Acc': f'{running_acc:.3f}', 'AUROC': f'{running_auroc:.3f}'})

    all_labels = np.concatenate(all_labels)
    all_preds  = np.concatenate(all_preds)
    all_probs  = np.concatenate(all_probs)
    return all_labels, all_preds, all_probs, all_paths

results = {}
print("\nEvaluating splits:")
for split, dl in [('dev_seen', dl_val), ('test_seen', dl_test),
                  ('dev_unseen', dl_val_unseen), ('test_unseen', dl_test_unseen)]:
    lbl, pred, prob, paths = collect_predictions(best_model, dl, split)
    valid = lbl >= 0
    results[split] = {
        'labels': lbl[valid], 'preds': pred[valid], 'probs': prob[valid],
        'all_labels': lbl, 'all_preds': pred, 'all_probs': prob, 'paths': paths,
    }
    if valid.sum() > 0:
        acc = (lbl[valid] == pred[valid]).mean()
        auc = roc_auc_score(lbl[valid], prob[valid])
        p, r, f, _ = precision_recall_fscore_support(lbl[valid], pred[valid], average='binary')
        print(f"  {split:15s}  Acc={acc:.4f}  AUROC={auc:.4f}  F1={f:.4f}  P={p:.4f}  R={r:.4f}")

print("\n✅ Evaluation complete.")


Loading best checkpoint: /kaggle/working/checkpoints/issues-layout_hmc_issues_lilt_lora_v1-epoch=08-val/auroc=0.7533.ckpt


Loading weights:   0%|          | 0/400 [00:00<?, ?it/s]

   LiLT: Froze 124.65M text-side params
   LiLTLayoutBranch (LoRA r=2): 0.018M / 130.77M params trainable (0.01%)
✅ Loaded pre-trained ISSUES projection weights into image_map & text_map

Evaluating splits:


  dev_seen         Acc=0.6840  AUROC=0.7532  F1=0.6565  P=0.7089  R=0.6113


  test_seen        Acc=0.7030  AUROC=0.7900  F1=0.6810  P=0.7188  R=0.6469


  dev_unseen       Acc=0.6852  AUROC=0.7333  F1=0.5771  P=0.5743  R=0.5800


  test_unseen      Acc=0.6900  AUROC=0.7582  F1=0.5899  P=0.5853  R=0.5947

✅ Evaluation complete.


## 📊 Research-Grade Evaluation Plots

In [20]:
import seaborn as sns

ACCENT  = '#7c6af7'; ACCENT2 = '#f7706a'; GREEN = '#4ade80'; YELLOW = '#facc15'
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#3a3a5c',   'axes.labelcolor': '#e0e0ff',
    'xtick.color': '#a0a0cc',      'ytick.color': '#a0a0cc',
    'text.color': '#e0e0ff',       'grid.color': '#2a2a4a',
    'grid.linestyle': '--',        'grid.alpha': 0.5,
    'font.family': 'monospace',    'axes.grid': True, 'figure.dpi': 120,
})
split_colors = {'dev_seen': ACCENT, 'test_seen': ACCENT2,
                'dev_unseen': GREEN, 'test_unseen': YELLOW}

fig = plt.figure(figsize=(22, 14))
fig.suptitle(f'Evaluation Report — {run_name}', fontsize=14, fontweight='bold', color='#e0e0ff', y=1.01)
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.4)

ax_roc = fig.add_subplot(gs[0, :2])
ax_roc.plot([0,1],[0,1],'gray',lw=1,ls='--',label='Random')
for split, res in results.items():
    fpr,tpr,_ = roc_curve(res['labels'], res['probs'])
    auc = roc_auc_score(res['labels'], res['probs'])
    ax_roc.plot(fpr, tpr, color=split_colors[split], lw=2.5, label=f'{split} ({auc:.3f})')
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.set_title('ROC Curves', fontweight='bold'); ax_roc.legend(framealpha=0.2, fontsize=8)

ax_pr = fig.add_subplot(gs[0, 2:])
for split, res in results.items():
    prec,rec,_ = precision_recall_curve(res['labels'], res['probs'])
    ap = average_precision_score(res['labels'], res['probs'])
    ax_pr.plot(rec, prec, color=split_colors[split], lw=2.5, label=f'{split} (AP={ap:.3f})')
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title('Precision-Recall Curves', fontweight='bold'); ax_pr.legend(framealpha=0.2, fontsize=8)

for idx, (split, res) in enumerate(results.items()):
    ax = fig.add_subplot(gs[1, idx])
    cm = confusion_matrix(res['labels'], res['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Hate','Hate'], yticklabels=['Not Hate','Hate'],
                linewidths=2, linecolor='#0f0f1a', cbar=False,
                annot_kws={'size':12,'weight':'bold','color':'white'})
    ax.set_title(f'{split}\nAcc={(res["labels"]==res["preds"]).mean():.3f}',
                 fontweight='bold', fontsize=9)

ax_bar = fig.add_subplot(gs[2, :])
metric_data = {s: {
    'Accuracy': (r['labels']==r['preds']).mean(),
    'AUROC':    roc_auc_score(r['labels'], r['probs']),
    'AP':       average_precision_score(r['labels'], r['probs']),
} for s, r in results.items()}
x = np.arange(len(results)); w = 0.25
for i, (met, col) in enumerate(zip(['Accuracy','AUROC','AP'], [ACCENT, ACCENT2, GREEN])):
    vals = [metric_data[s][met] for s in results]
    bars = ax_bar.bar(x+i*w, vals, w, label=met, color=col, edgecolor='none')
    for bar, v in zip(bars, vals):
        ax_bar.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7, color='#e0e0ff')
ax_bar.set_xticks(x+w); ax_bar.set_xticklabels(list(results.keys()), fontsize=8)
ax_bar.set_ylim(0, 1.1); ax_bar.set_title('Metrics by Split', fontweight='bold')
ax_bar.legend(framealpha=0.2)

plt.savefig(f'{args.output_dir}/{run_name}_evaluation.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("📊 Evaluation plots saved.")

print("\n" + "="*65)
print("  CLASSIFICATION REPORTS")
print("="*65)
for split, res in results.items():
    print(f"\n  {split}")
    print("-"*50)
    print(classification_report(res['labels'], res['preds'],
          target_names=['Not Hateful', 'Hateful'], digits=4))


<Figure size 2640x1680 with 7 Axes>

📊 Evaluation plots saved.

  CLASSIFICATION REPORTS

  dev_seen
--------------------------------------------------
              precision    recall  f1-score   support

 Not Hateful     0.6655    0.7549    0.7074       253
     Hateful     0.7089    0.6113    0.6565       247

    accuracy                         0.6840       500
   macro avg     0.6872    0.6831    0.6820       500
weighted avg     0.6870    0.6840    0.6823       500


  test_seen
--------------------------------------------------
              precision    recall  f1-score   support

 Not Hateful     0.6905    0.7569    0.7222       510
     Hateful     0.7188    0.6469    0.6810       490

    accuracy                         0.7030      1000
   macro avg     0.7047    0.7019    0.7016      1000
weighted avg     0.7044    0.7030    0.7020      1000


  dev_unseen
--------------------------------------------------
              precision    recall  f1-score   support

 Not Hateful     0.7515    0.7471    0.7493    

## 🔍 Inference Visualization

In [21]:
def visualize_inference(results_split, split='dev_seen', seed=42):
    random.seed(seed)
    res = results_split
    df_inf = pd.DataFrame({'path': res['paths'], 'label': res['all_labels'],
                           'pred': res['all_preds'], 'prob': res['all_probs']})
    df_inf = df_inf[df_inf['label'] >= 0].reset_index(drop=True)
    categories = {
        'TP': df_inf[(df_inf.label==1)&(df_inf.pred==1)],
        'TN': df_inf[(df_inf.label==0)&(df_inf.pred==0)],
        'FP': df_inf[(df_inf.label==0)&(df_inf.pred==1)],
        'FN': df_inf[(df_inf.label==1)&(df_inf.pred==0)],
    }
    sample_df = pd.concat([
        cat.sample(min(4, len(cat)), random_state=seed)
        for cat in categories.values()
    ]).reset_index(drop=True)
    n_cols = 4; n_rows = int(np.ceil(len(sample_df) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4.5, n_rows*5.5))
    fig.suptitle(f'Inference: GT vs Prediction — {split}', fontsize=14,
                 fontweight='bold', color='#e0e0ff', y=1.01)
    axes = axes.flatten()
    STATUS_COLORS = {'TP': GREEN, 'TN': GREEN, 'FP': ACCENT2, 'FN': YELLOW}
    LABEL_MAP     = {0: 'Not Hateful', 1: 'Hateful'}
    for i, (ax, (_, row)) in enumerate(zip(axes, sample_df.iterrows())):
        try:
            ax.imshow(Image.open(row['path']).convert('RGB').resize((320, 320)))
        except Exception:
            ax.set_facecolor('#1a1a2e')
        status = {(1,1):'TP',(0,0):'TN',(0,1):'FP',(1,0):'FN'}.get(
            (int(row['label']), int(row['pred'])), '?')
        color = STATUS_COLORS.get(status, 'gray')
        ax.set_title(
            f"GT: {LABEL_MAP[int(row['label'])]}\nPred: {LABEL_MAP[int(row['pred'])]} [{status}]\nConf: {row['prob']:.2f}",
            fontsize=8.5, color=color, fontweight='bold', pad=4)
        for spine in ax.spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(3)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes[len(sample_df):]: ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{args.output_dir}/{run_name}_inference_{split}.png',
                bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()
    print(f"📸 Inference grid saved for {split}")

visualize_inference(results['dev_seen'],  split='dev_seen')
visualize_inference(results['test_seen'], split='test_seen')


<Figure size 2160x2640 with 16 Axes>

📸 Inference grid saved for dev_seen


<Figure size 2160x2640 with 16 Axes>

📸 Inference grid saved for test_seen


## 💾 Save Experiment Summary

In [22]:
log_df = pd.read_csv(log_csv_path)
exp_summary = {
    'run_name':     run_name,
    'model':        args.name,
    'layout_model': 'LiLT (SCUT-DLVCLab/lilt-roberta-en-base)',
    'map_dim':      args.map_dim,
    'layout_dim':   args.layout_dim,
    'lora_r':       args.lora_r,
    'lora_alpha':   args.lora_alpha,
    'lora_target':  args.lora_target_modules,
    'lilt_text_frozen': True,
    'phi_frozen':   True,
    'from_scratch': True,
    'lr':           args.lr,
    'batch_size':   args.batch_size,
    'best_epoch':   int(log_df.loc[log_df['val_auroc'].dropna().idxmax(), 'epoch']),
    'results': {
        split: {
            'accuracy': float((res['labels']==res['preds']).mean()),
            'auroc':    float(roc_auc_score(res['labels'], res['probs'])),
            'ap':       float(average_precision_score(res['labels'], res['probs'])),
        } for split, res in results.items()
    },
    'epoch_log_csv': log_csv_path,
    'best_ckpt':     best_ckpt,
}

json_path = f'{args.experiment_log_dir}/{run_name}_summary.json'
with open(json_path, 'w') as f:
    json.dump(exp_summary, f, indent=2)

print("\n" + "="*65)
print(f"  FINAL RESULTS — {run_name}")
print("="*65)
for split, metrics in exp_summary['results'].items():
    print(f"  {split:15s}  Acc={metrics['accuracy']:.4f}  "
          f"AUROC={metrics['auroc']:.4f}  AP={metrics['ap']:.4f}")
print("="*65)
print(f"  Best Epoch  : {exp_summary['best_epoch']}")
print(f"  Checkpoint  : {best_ckpt}")
print(f"  Layout Model: LiLT (lilt-roberta-en-base)")
print(f"  LoRA rank   : r={args.lora_r}, alpha={args.lora_alpha}")
print(f"  LiLT text   : Frozen (RoBERTa side)")
print(f"  Phi frozen  : True (weights used as-is)")
print("="*65)
print(f"✅ Summary → {json_path}")



  FINAL RESULTS — issues-layout_hmc_issues_lilt_lora_v1
  dev_seen         Acc=0.6840  AUROC=0.7532  AP=0.7202
  test_seen        Acc=0.7030  AUROC=0.7900  AP=0.7627
  dev_unseen       Acc=0.6852  AUROC=0.7333  AP=0.5826
  test_unseen      Acc=0.6900  AUROC=0.7582  AP=0.6438
  Best Epoch  : 9
  Checkpoint  : /kaggle/working/checkpoints/issues-layout_hmc_issues_lilt_lora_v1-epoch=08-val/auroc=0.7533.ckpt
  Layout Model: LiLT (lilt-roberta-en-base)
  LoRA rank   : r=2, alpha=8
  LiLT text   : Frozen (RoBERTa side)
  Phi frozen  : True (weights used as-is)
✅ Summary → /kaggle/working/experiments/issues-layout_hmc_issues_lilt_lora_v1_summary.json
